# World generator (library)

Builds the Gazebo world `gambit_chess` (`gambit_bringup/world/chess_world.sdf`) from the
single source of truth `gambit_bringup/config/board.yaml`. Every other sim notebook
(`teleport`, `data_generator`, `fault_injector`) `%run`s this one for the board geometry:
square centers, piece heights, model instance names and the graveyard row.

**Deliverable** (README "Updated plan (11/4/2026)"): *Gazebo world, pieces in Gazebo*.
The world file in the repo was produced by an earlier generator; this notebook reproduces
its piece/board/camera `<include>` poses exactly (the self-check parses the SDF and compares).

**TODO checklist**
- [x] square-center formula from board.yaml (matches docs/gazebo.md)
- [x] model instance names for the start position + spare queens in the graveyard
- [x] camera pose (roll/pitch/yaw) from `position` + `look_at`
- [x] world SDF text (table, light, board, camera, 36 piece includes)
- [ ] `piece_model_sdf()`: generate the 12 piece models (visual stack per piece type)
- [ ] `board_texture()`: regenerate the board texture/mesh with the ArUco corner markers
- [ ] try `static: false` pieces once teleport works (physical grasping)

In [ ]:
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import yaml

REPO = Path("../..").resolve()
BOARD_YAML = REPO / "gambit_bringup/config/board.yaml"
WORLD_SDF = REPO / "gambit_bringup/world/chess_world.sdf"
CAMERA_SDF = REPO / "gambit_bringup/models/board_camera/model.sdf"
WORLD_NAME = "gambit_chess"

def load_board_cfg(path=BOARD_YAML):
    with open(path) as f:
        return yaml.safe_load(f)

BOARD = load_board_cfg()

## Square centers

World frame: origin at the arm base, x toward the board, y to the arm's left, z up; the
table top is z = 0. With `white_near_arm: true`, rank 1 is nearest the arm, so

    x = cx + (rank - 3.5) * s        y = cy + (3.5 - file) * s        z = board thickness

with file 0 = a, rank 0 = rank 1, s = square edge. (docs/gazebo.md: x = 0.215 + (rank-3.5)*0.025,
y = (3.5-file)*0.025, z = 0.006.) The a-file is on +y (the arm's left) when viewed from the arm.

In [ ]:
FILES = "abcdefgh"

def square_center(square, cfg=BOARD):
    """'e4' (or python-chess index 0..63) -> (x, y, z) of the square center on the board top."""
    if isinstance(square, int):
        f, r = square % 8, square // 8
    else:
        f, r = FILES.index(square[0]), int(square[1]) - 1
    b = cfg["board"]
    s = b["square"]
    cx, cy = b["center"]
    if not b.get("white_near_arm", True):
        f, r = 7 - f, 7 - r          # board rotated 180 degrees: h8 nearest the arm
    return (cx + (r - 3.5) * s, cy + (3.5 - f) * s, b["thickness"])

def board_outer_size(cfg=BOARD):
    b = cfg["board"]
    return 8 * b["square"] + 2 * b["border"]

def marker_centers(cfg=BOARD):
    """ArUco marker centers {id: (x, y)}: border/2 outside the playing-area corner on both axes."""
    b, s = cfg["board"], cfg["board"]["square"]
    out = {}
    for corner, mid in cfg["markers"]["ids"].items():
        x, y, _ = square_center(corner, cfg)
        sx = 1 if x > b["center"][0] else -1
        sy = 1 if y > b["center"][1] else -1
        out[mid] = (x + sx * (s / 2 + b["border"] / 2), y + sy * (s / 2 + b["border"] / 2))
    return out

## Pieces: sizes and model names

Piece height and base radius are fractions of the square edge (`pieces.height`, `pieces.radius`).
Model *types* are `gambit_piece_<color><KIND>` (e.g. `gambit_piece_wP`); model *instances* in the
world are `<color>_<KIND>_<n>`: pawns numbered by file (`w_P_1` = a-pawn ... `w_P_8` = h-pawn),
pieces numbered a-side first (`w_R_1` on a1, `w_R_2` on h1), queen/king `_1`. Spare queens
`_2`, `_3` start in the graveyard: a row along +x starting at `pieces.graveyard`, spaced
`GRAVEYARD_SPACING` (0.9 squares, matching the repo world), on the table (z = 0).

In [ ]:
KINDS = "PNBRQK"
BACK_RANK = "RNBQKBNR"
GRAVEYARD_SPACING = 0.0225     # 0.9 * square in the repo world; keep in sync if square changes

def piece_height(kind, cfg=BOARD):
    return cfg["pieces"]["height"][kind.upper()] * cfg["board"]["square"]

def piece_radius(cfg=BOARD):
    return cfg["pieces"]["radius"] * cfg["board"]["square"]

def model_type(color, kind):
    """('w', 'P') -> 'gambit_piece_wP'."""
    return f"gambit_piece_{color}{kind.upper()}"

def instance_name(color, kind, n):
    return f"{color}_{kind.upper()}_{n}"

def graveyard_pose(slot, cfg=BOARD):
    gx, gy = cfg["pieces"]["graveyard"]
    return (gx + slot * GRAVEYARD_SPACING, gy, 0.0)

def start_instances(cfg=BOARD):
    """Ordered list of (instance_name, model_type, square or None) exactly as in the repo world:
    white back rank, white pawns, black pawns, black back rank, then spare queens (square None)."""
    out = []
    def back(color, rank):
        seen = {}
        for f, k in enumerate(BACK_RANK):
            seen[k] = seen.get(k, 0) + 1
            out.append((instance_name(color, k, seen[k]), model_type(color, k), f"{FILES[f]}{rank}"))
    def pawns(color, rank):
        for f in range(8):
            out.append((instance_name(color, "P", f + 1), model_type(color, "P"), f"{FILES[f]}{rank}"))
    back("w", 1); pawns("w", 2); pawns("b", 7); back("b", 8)
    for color in "wb":
        for i in range(cfg["pieces"]["spare_queens"]):
            out.append((instance_name(color, "Q", i + 2), model_type(color, "Q"), None))
    return out

def start_poses(cfg=BOARD):
    """{instance_name: (x, y, z)} for the start position; spare queens fill graveyard slots 0, 1, ..."""
    poses, slot = {}, 0
    for name, _, sq in start_instances(cfg):
        if sq is None:
            poses[name] = graveyard_pose(slot, cfg); slot += 1
        else:
            poses[name] = square_center(sq, cfg)
    return poses

## Camera pose

The camera model's sensor looks along its +x axis. To point it from `position` at `look_at`:
yaw = atan2(dy, dx), pitch = atan2(-dz, sqrt(dx^2 + dy^2)) (positive pitch tilts +x downward in
SDF's roll-pitch-yaw convention), roll = 0.

In [ ]:
def camera_rpy(cfg=BOARD):
    px, py, pz = cfg["camera"]["position"]
    lx, ly, lz = cfg["camera"]["look_at"]
    dx, dy, dz = lx - px, ly - py, lz - pz
    return (0.0, math.atan2(-dz, math.hypot(dx, dy)), math.atan2(dy, dx))

def camera_pose(cfg=BOARD):
    return tuple(cfg["camera"]["position"]) + camera_rpy(cfg)

## SDF text

Small string templates. `include_xml` is the piece/board/camera snippet; `world_sdf_text`
assembles the full world (physics + system plugins, scene, sun, table, includes).

In [ ]:
def fmt_pose(p):
    """Pose tuple -> SDF text. 3 numbers get ' 0 0 0' appended (format used in the repo world)."""
    if len(p) == 3:
        return f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f} 0 0 0"
    return " ".join(f"{v:.4f}" for v in p)

def include_xml(name, uri, pose_text, indent="    "):
    return (f"{indent}<include>\n{indent}  <name>{name}</name>\n{indent}  <uri>model://{uri}</uri>\n"
            f"{indent}  <pose>{pose_text}</pose>\n{indent}</include>\n")

WORLD_HEADER = """<?xml version="1.0"?>
<!-- Generated by gambit_sim/notebooks/world_generator.ipynb from gambit_bringup/config/board.yaml.
     Do not edit by hand: change board.yaml and run the generator again. -->
<sdf version="1.9">
  <world name="{world}">
    <physics name="1ms" type="ignored">
      <max_step_size>0.001</max_step_size>
      <real_time_factor>1.0</real_time_factor>
    </physics>
    <plugin filename="gz-sim-physics-system" name="gz::sim::systems::Physics"/>
    <plugin filename="gz-sim-user-commands-system" name="gz::sim::systems::UserCommands"/>
    <plugin filename="gz-sim-scene-broadcaster-system" name="gz::sim::systems::SceneBroadcaster"/>
    <plugin filename="gz-sim-sensors-system" name="gz::sim::systems::Sensors">
      <render_engine>ogre2</render_engine>
    </plugin>

    <gravity>0 0 -9.81</gravity>
    <scene>
      <ambient>0.6 0.6 0.6 1</ambient>
      <background>0.75 0.75 0.78 1</background>
      <shadows>true</shadows>
    </scene>

    <light type="directional" name="sun">
      <cast_shadows>true</cast_shadows>
      <pose>0.3 0.2 2.0 0 0 0</pose>
      <diffuse>0.85 0.85 0.85 1</diffuse>
      <specular>0.2 0.2 0.2 1</specular>
      <direction>-0.3 -0.2 -1.0</direction>
    </light>

    <model name="table">
      <static>true</static>
      <link name="link">
        <collision name="collision">
          <pose>0.2 0 -0.01 0 0 0</pose>
          <geometry><box><size>1.2 1.2 0.02</size></box></geometry>
        </collision>
        <visual name="visual">
          <pose>0.2 0 -0.01 0 0 0</pose>
          <geometry><box><size>1.2 1.2 0.02</size></box></geometry>
          <material><ambient>0.55 0.55 0.58 1</ambient><diffuse>0.55 0.55 0.58 1</diffuse></material>
        </visual>
      </link>
    </model>

"""
WORLD_FOOTER = "  </world>\n</sdf>\n"

def world_sdf_text(cfg=BOARD, world=WORLD_NAME):
    cx, cy = cfg["board"]["center"]
    parts = [WORLD_HEADER.format(world=world)]
    parts.append(include_xml("chess_board", "chess_board", f"{cx:.4f} {cy:.4f} 0 0 0 0"))
    parts.append(include_xml("board_camera", "board_camera", fmt_pose(camera_pose(cfg))))
    poses = start_poses(cfg)
    for name, mtype, _ in start_instances(cfg):
        x, y, z = poses[name]
        parts.append(include_xml(name, mtype, f"{x:.4f} {y:.4f} {z:.4f} 0 0 0"))
    parts.append(WORLD_FOOTER)
    return "".join(parts)

def write_world(path=WORLD_SDF, cfg=BOARD):
    """Overwrite the world file (only call from a run cell, never from the self-check)."""
    Path(path).write_text(world_sdf_text(cfg))

## Piece and board models (TODO)

The 12 piece models in `gambit_bringup/models/pieces/` are stacks of cylinders/spheres
(see `gambit_piece_wP/model.sdf`: base disc, stem, head) plus one collision cylinder of
radius `piece_radius()` and height `piece_height(kind)`. Regenerating them lets you change
board.yaml (bigger squares, other colors) and keep everything consistent.

In [ ]:
def piece_model_sdf(color, kind, cfg=BOARD):
    """Return the model.sdf text for `gambit_piece_<color><kind>`.

    TODO: design the visual stack per piece type (the knight is the hard one: a tilted box
    for the head works; a mesh is nicer). Keep: one collision cylinder (radius piece_radius(),
    length piece_height(kind), origin at the base so the include pose z = board top),
    inertial with a plausible mass, material from pieces.white_rgba / black_rgba,
    <static> from pieces.static. Compare against the existing files for the format.
    """
    raise NotImplementedError("TODO: piece_model_sdf")

def board_texture(cfg=BOARD, px_per_square=128):
    """Return an RGB numpy image of the board top (squares + frame + ArUco markers).

    TODO: draw light/dark squares (board.light_rgb/dark_rgb), the frame (frame_rgb), and the
    4 markers with cv2.aruco.generateImageMarker(DICT_4X4_50, id, size) at marker_centers().
    Remember a1 must be dark and image orientation must match the mesh UVs in board_top.obj.
    """
    raise NotImplementedError("TODO: board_texture")

## Self-check
Parse the repo world and compare every `<include>` pose with ours; check the numbers from
docs/gazebo.md.

In [ ]:
def parse_world_includes(path=WORLD_SDF):
    root = ET.parse(path).getroot()
    out = {}
    for inc in root.iter("include"):
        out[inc.findtext("name")] = (inc.findtext("uri"), [float(v) for v in inc.findtext("pose").split()])
    return out

def _close(a, b, tol=1e-4):
    return all(abs(x - y) < tol for x, y in zip(a, b))

assert _close(square_center("e4"), (0.2025, -0.0125, 0.006))
assert _close(square_center("a1"), (0.1275, 0.0875, 0.006))
assert _close(square_center(63), square_center("h8"))
assert abs(board_outer_size() - 0.26) < 1e-9
assert abs(piece_height("P") - 0.0225) < 1e-9 and abs(piece_radius() - 0.0085) < 1e-9

world_incs = parse_world_includes()
assert len(start_instances()) == 36 and len(world_incs) == 38
for name, mtype, _ in start_instances():
    uri, pose = world_incs[name]
    assert uri == f"model://{mtype}", name
    assert _close(pose[:3], start_poses()[name]), (name, pose, start_poses()[name])
assert _close(world_incs["board_camera"][1], camera_pose()), camera_pose()
assert _close(world_incs["chess_board"][1][:2], BOARD["board"]["center"])

gen = ET.fromstring(world_sdf_text().split("\n", 1)[1])  # skip the XML declaration
assert gen.find("world").get("name") == WORLD_NAME
assert len(list(gen.iter("include"))) == 38
print("world_generator self-check OK:", len(world_incs), "includes match")